# V50 - Diverse candidates under template-grouped gates

Goal: beat the frozen 0.66028 champion by fixing the diagnosed bottleneck, not by adding another similar ranker. Evidence: v30/v33/champion agree on 99.7% of test top-1, and 16.7% of unique chat texts are exact duplicates shared across train and test. So the next gain must come from candidates with a different inductive bias, validated under template-grouped folds with fail-close gates.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 20260905
N_SPLITS = 5
GATE_FLOOR = 0.00045
rng = np.random.RandomState(SEED)

# Portable paths: Kaggle input dataset, else local data folder.
CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found in any candidate path"
CHAMPION = next((p for p in [Path("/kaggle/input/intelligo-champion/submission_v42_primary.csv"),
                            Path("results/v42/submission_v42_primary.csv")] if p.exists()), None)
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v50")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve(), "| CHAMPION:", CHAMPION, "| OUT:", OUT.resolve())

## Load data and enforce entity disjointness

Train, test, and assessment users must be disjoint entities. Chat may overlap by template, which is measured in the next cell, not assumed away.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")

train_ids = set(train["user_id"])
assert not (train_ids & set(test_ids)), "train/test user overlap"
assert train_ids <= set(assess["user_id"]), "train users missing assessments"
assert set(test_ids) <= set(assess["user_id"]), "test users missing assessments"
print(f"train={len(train)} test={len(test_ids)} chats={len(chats)} assessed={len(assess)}")
Y = train[MODS].to_numpy(float)
print(f"nonzero label fraction: {(Y > 0).mean():.3f}")

## Template leakage audit

Chat texts come from templates. Exact-duplicate texts shared across train and test users are counted, then each user is assigned a template group used for grouped folds.

In [ ]:
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
owners = {}
for u, ts in by_user.items():
    for t in ts:
        owners.setdefault(t, set()).add(u)
shared = sum(1 for us in owners.values() if (us & train_ids) and (us & set(test_ids)))
print(f"unique texts={len(owners)} shared train<->test={shared} ({shared / len(owners):.3f})")

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    head = " ".join(str(texts[0]).lower().split()[:6])
    return re.sub(r"\d+", "#", head)

all_users = list(train["user_id"]) + test_ids
groups = {u: template_group(by_user.get(u, [])) for u in all_users}
print("NO_CHAT users:", sum(1 for g in groups.values() if g == "NO_CHAT"))

## Features: assessment matrix and chat TF-IDF

Two deliberately different views: assessment-only (immune to template noise) and chat-only (semantic). Their disagreement is the point.

In [ ]:
blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan

A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
col_mean = np.nanmean(A, axis=0)
A = np.where(np.isnan(A), col_mean, A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
print("assessment matrix:", A.shape, f"({len(questions)} questions)")

docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3)
T = tfidf.fit_transform(docs)
print("tfidf:", T.shape)

n_train = len(train)
A_tr, A_te = A[:n_train], A[n_train:]
T_tr, T_te = T[:n_train], T[n_train:]
strata = Y.argmax(1)
fold_groups = np.array([groups[u] for u in train["user_id"]])

## Grouped OOF evaluation of three diverse baselines

Popularity, assessment-only Ridge, chat-only Ridge. Folds are stratified by top module and grouped by chat template, so template leakage cannot inflate the numbers.

In [ ]:
def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def ridge_oof(Xtr, Xte, alpha=3.0):
    pred = np.zeros((Xte.shape[0], len(MODS)))
    oof = np.zeros_like(Y)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_score = []
    fold_id = np.zeros(len(Xtr), int)
    for fi, (tr, va) in enumerate(skf.split(Xtr, strata, fold_groups)):
        p_va = np.zeros((len(va), len(MODS)))
        for j in range(len(MODS)):
            m = Ridge(alpha=alpha, random_state=SEED)
            m.fit(Xtr[tr], Y[tr, j])
            p_va[:, j] = m.predict(Xtr[va])
            if Xte is not None:
                pred[:, j] += m.predict(Xte) / N_SPLITS
        oof[va] = p_va
        fold_score.append(ndcg5(Y[va], p_va).mean())
        fold_id[va] = fi + 1
    return oof, pred, np.array(fold_score), fold_id

pop_oof = np.tile(Y.mean(0), (n_train, 1))
pop_folds = []
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for tr, va in skf.split(A_tr, strata, fold_groups):
    pop_folds.append(ndcg5(Y[va], np.tile(Y[tr].mean(0), (len(va), 1))).mean())
print(f"popularity OOF: {np.mean(pop_folds):.5f} folds={np.round(pop_folds, 5)}")

oof_a, pred_a, f_a, fid = ridge_oof(A_tr, A_te)
print(f"assessment-Ridge OOF: {ndcg5(Y, oof_a).mean():.5f} folds={np.round(f_a, 5)}")
oof_t, pred_t, f_t, _ = ridge_oof(T_tr.toarray(), T_te.toarray())
print(f"chat-Ridge OOF: {ndcg5(Y, oof_t).mean():.5f} folds={np.round(f_t, 5)}")

## Disagreement and gated combination

The candidates are combined only where they genuinely disagree. A paired bootstrap CI on OOF user scores decides: adopt the blend, or fail close to the stronger single view.

In [ ]:
agree = (oof_a.argmax(1) == oof_t.argmax(1)).mean()
print(f"assessment vs chat top-1 agreement on OOF: {agree:.3f} (low is good here)")

ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]
oof_scores = {"popularity": ndcg5(Y, pop_oof), "assess": ndcg5(Y, oof_a), "chat": ndcg5(Y, oof_t)}
best = ("assess", oof_scores["assess"])
for name, s in oof_scores.items():
    if s.mean() > best[1].mean():
        best = (name, s)
print("best single:", best[0], f"{best[1].mean():.5f}")

blend_oof, blend_pred, f_b = None, None, None
for a in ALPHAS[1:-1]:
    bo = a * oof_a + (1 - a) * oof_t
    bp = a * pred_a + (1 - a) * pred_t
    d = ndcg5(Y, bo) - best[1]
    lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
    wf = min(ndcg5(Y[fid == k], bo[fid == k]).mean() - best[1][fid == k].mean() for k in range(1, N_SPLITS + 1))
    print(f"alpha={a}: delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
    if lo > GATE_FLOOR and (blend_oof is None or d.mean() > (ndcg5(Y, blend_oof) - best[1]).mean()):
        blend_oof, blend_pred = bo, bp

if blend_oof is None:
    print(f"FAIL-CLOSE: no blend clears gate {GATE_FLOOR}; keeping", best[0])
    final_oof, final_pred = best[1], {"popularity": np.tile(Y.mean(0), (len(test_ids), 1)), "assess": pred_a, "chat": pred_t}[best[0]]
else:
    print(f"ADOPT blend OOF: {ndcg5(Y, blend_oof).mean():.5f}")
    final_oof, final_pred = blend_oof, blend_pred

## Champion disagreement (optional input) and submission

If the frozen champion file is provided, report where the gated candidate disagrees with it. The output submission is written in the required 17-column format.

In [ ]:
if CHAMPION is not None:
    ch = pd.read_csv(CHAMPION)
    assert ch["user_id"].tolist() == test_ids, "champion row order mismatch"
    C = ch[MODS].to_numpy(float)
    t1 = (C.argmax(1) == final_pred.argmax(1)).mean()
    t5 = np.mean([len(set(a) & set(b)) / 5 for a, b in zip(np.argsort(-C, 1)[:, :5], np.argsort(-final_pred, 1)[:, :5])])
    print(f"candidate vs champion: top1 agree={t1:.3f} top5 overlap={t5:.3f}")
else:
    print("champion file absent; skipping disagreement report")

sub = pd.DataFrame(final_pred, columns=MODS)
sub.insert(0, "user_id", test_ids)
sub.to_csv(OUT / "submission_v50_primary.csv", index=False)
print("wrote", OUT / "submission_v50_primary.csv", sub.shape)
print(f"final gated OOF NDCG@5: {final_oof.mean():.5f}")